# Body Language Reader — YOLOv8 Pose Training 
**TellTrace Project 

This notebook fine-tunes YOLOv8 for pose detection, taking advantage of
Colab Pro's faster GPUs (V100/A100) and longer session limits.


---

## Cell 1 — Install dependencies

In [ ]:
!pip install ultralytics==8.3.0 -q
!pip install matplotlib opencv-python-headless -q

import ultralytics
ultralytics.checks()
print("\n All packages installed")

## Cell 2 — Mount Google Drive
Your trained model weights will be saved here so you don't lose them when the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/BodyLanguageReader'
os.makedirs(f'{PROJECT_DIR}/weights', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/datasets', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/runs', exist_ok=True)

print(f" Drive mounted. Project folder: {PROJECT_DIR}")
print(f" Weights will be saved to: {PROJECT_DIR}/weights/")

## Cell 3 — Dataset Setup

Using COCO Keypoints 2017 — no annotation needed, downloads automatically on first training run.

In [ ]:
DATA_YAML = 'coco-pose.yaml'   # Ultralytics built-in — auto-downloads on first run
print(" COCO-pose dataset selected")
print("  64k images — downloads automatically when training starts.")

## Cell 4 — Configure Training


In [ ]:
CONFIG = {
    # Model size: 'n'=nano, 's'=small (recommended), 'm'=medium, 'l'=large
    # With Colab Pro's faster GPU, 'm' is also a realistic option if you want
    # higher accuracy — it just takes proportionally longer.
    'model_size': 's',

    # Full epoch count — Colab Pro's GPU makes this fast
    'epochs': 10,

    # Image size in pixels (640 is standard for YOLOv8)
    'imgsz': 640,

    # Larger batch size — V100/A100 have more VRAM than the free T4 (16GB/40GB vs 12GB)
    'batch': 32,

    # Frozen layers: 10=fine-tuning (recommended), 0=full retrain
    'freeze': 10,

    # Name for this training run
    'name': 'body-language-v1',

    # Dataset config from Cell 4
    'data': 'coco-pose.yaml',
}

MODEL_NAME = f"yolov8{CONFIG['model_size']}-pose.pt"

print("Training configuration:")
for k, v in CONFIG.items():
    print(f"  {k:12s} : {v}")

print("\n  Estimated time on V100  : ~25-35 minutes")
print("  Estimated time on A100  : ~15-20 minutes")
print("\n Config ready. Run Cell 6 to start training.")

## Cell 5 — Train the Model
This is the main training cell.


In [ ]:
from ultralytics import YOLO
import time

print(f"Loading pretrained model: {MODEL_NAME}")
model = YOLO(MODEL_NAME)   # downloads ~22MB on first run

print("\nStarting training...")
print("Watch the pose_loss column — aim for it to drop below 1.0")
print("-" * 60)

start_time = time.time()

results = model.train(
    data          = CONFIG['data'],
    epochs        = CONFIG['epochs'],
    imgsz         = CONFIG['imgsz'],
    batch         = CONFIG['batch'],
    device        = 0,
    freeze        = CONFIG['freeze'],
    name          = CONFIG['name'],
    project       = '/content/runs/pose',
    pretrained    = True,
    exist_ok      = True,

    # Optimizer (tuned for fine-tuning)
    lr0           = 0.001,
    lrf           = 0.01,
    warmup_epochs = 3,

    # Augmentation (tuned for interview / meeting scenarios)
    mosaic        = 1.0,
    flipud        = 0.0,     # people are always upright
    degrees       = 10.0,    # slight rotation
    hsv_v         = 0.4,     # brightness variation
    hsv_h         = 0.015,
    close_mosaic  = 10,      # stable final epochs

    plots         = True,
    verbose       = True,
)

elapsed = (time.time() - start_time) / 60
print(f"\n Training complete in {elapsed:.1f} minutes")

RUN_DIR = f"/content/runs/pose/{CONFIG['name']}" 

## Cell 6 — Save weights to Google Drive

In [ ]:
import shutil, os
from datetime import datetime

RUN_DIR  = f"/content/runs/pose/{CONFIG['name']}"
BEST_PT  = f"{RUN_DIR}/weights/best.pt"
LAST_PT  = f"{RUN_DIR}/weights/last.pt"

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
SAVE_DIR  = f"{PROJECT_DIR}/weights/{CONFIG['name']}_{timestamp}"
os.makedirs(SAVE_DIR, exist_ok=True)

shutil.copy(BEST_PT, f"{SAVE_DIR}/best.pt")
shutil.copy(LAST_PT, f"{SAVE_DIR}/last.pt")

for f in ['results.png', 'results.csv', 'confusion_matrix.png', 'args.yaml']:
    src = f"{RUN_DIR}/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"{SAVE_DIR}/{f}")

print(f" Weights saved to Google Drive:")
print(f"   {SAVE_DIR}/best.pt  ← use this on your laptop")
print(f"\nFiles saved:")
for f in os.listdir(SAVE_DIR):
    size_mb = os.path.getsize(f"{SAVE_DIR}/{f}") / 1e6
    print(f"   {f:30s}  {size_mb:.1f} MB")

---
## After training — next steps

1. **Download `best.pt`** from Google Drive → `BodyLanguageReader/weights/`
2. **Copy to your laptop** → `body-language-reader/models/weights/best.pt`
3. **Use it in your pipeline** — in extract.py or classifier.py, change:
   ```python
   model = YOLO('yolov8n-pose.pt')
   ```
   to:
   ```python
   model = YOLO('models/weights/best.pt')
   ```

---
*TellTrace — AI Body Language Reader